# Mem0 Memory — Persistent User Memory Across Sessions
## Semantic Persistence — Facts That Outlive the Process
⏱ ~12 min

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Esturban/agent/blob/master/examples/67-mem0-memory/mem0_workbook.ipynb)

Most LLM conversations are stateless — the model forgets everything between sessions. **Mem0** adds a persistent memory layer: facts extracted from conversations are stored and retrieved by semantic search, giving agents genuine long-term memory. This workshop builds a two-session memory agent: Session 1 stores preferences about Alice, Session 2 recalls them to answer personalized questions — all via a LangGraph pipeline. By the end you will understand *how* Mem0 stores and retrieves memories and *how* to integrate it with LangGraph's state.

---

### Workshop Roadmap

| # | Topic |
|---|-------|
| 1 | **Concepts** — Stateless vs stateful agents, how Mem0 works |
| 2 | **Store Memories** — `mem.add()` from a conversation |
| 3 | **Recall Memories** — `mem.search()` by semantic similarity |
| 4 | **LangGraph Integration** — recall → respond → store pipeline |
| 5 | **Two-Session Demo** — Session 1 stores, Session 2 recalls |
| ★ | **Exercises + Answer Key** |

### Prerequisites
- Python 3.10+, or Google Colab (install cell below)
- `OPENAI_API_KEY` in `.env` or Colab Secrets

In [ ]:
import sys

def _in_colab():
    try:
        import google.colab; return True
    except ImportError:
        return False

if _in_colab():
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
        "mem0ai", "langchain", "langchain-openai", "langgraph", "python-dotenv"])
    print("Colab install complete.")
else:
    print("Local — skipping install.")

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    from dotenv import load_dotenv; load_dotenv()

key = os.environ.get("OPENAI_API_KEY", "")
print(f"API key ready: {bool(key) and key.startswith('sk-')}")

---
## Part 1 — Stateless vs Stateful Agents

| | Stateless | Mem0 (Stateful) |
|---|---|---|
| Memory scope | Current conversation only | Across all sessions |
| Storage | None | Vector DB (default: in-memory) |
| Personalization | None | User-specific semantic recall |
| Privacy risk | Low | Must manage per-user data carefully |

**How Mem0 works:**
1. `mem.add(messages, user_id=uid)` — extracts facts from conversation, embeds + stores them
2. `mem.search(query, user_id=uid)` — semantic search returns relevant memories as context

In [ ]:
from mem0 import Memory
from langchain_openai import ChatOpenAI

USER_ID = "demo-user-001"
# Memory() defaults to in-process vector store (hnswlib) — no external DB needed to get started
mem = Memory()
# temperature=0 for deterministic recall-based answers — raise to 0.7+ for more creative responses
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

print(f"Mem0 ready  |  User ID: {USER_ID}")

---
## Part 2 — Store Memories from a Conversation

In [ ]:
SESSION_1_MESSAGES = [
    {"role": "user",      "content": "Hi, I'm Alice. I prefer Python over Java."},
    {"role": "assistant", "content": "Got it, Alice! I'll keep Python in mind."},
    {"role": "user",      "content": "I also like concise code — no boilerplate."},
    {"role": "assistant", "content": "Noted. Concise and Pythonic it is."},
]

print("Storing Session 1 memories...")
# mem.add() sends the conversation to an LLM that extracts discrete facts, then embeds and stores them
# user_id scopes memories to this user — the same key is used for search isolation
mem.add(SESSION_1_MESSAGES, user_id=USER_ID)

# Inspect what was stored
all_memories = mem.get_all(user_id=USER_ID)
memories_list = all_memories.get("results", all_memories) if isinstance(all_memories, dict) else all_memories
print(f"\nStored {len(memories_list)} memories:")
for i, m in enumerate(memories_list, 1):
    print(f"  {i}. {m.get('memory', m.get('text', str(m)))}")

---
## Part 3 — Recall Memories by Semantic Search

In [ ]:
SESSION_2_QUERIES = [
    "What programming language do I prefer?",
    "What is my name?",
    "What coding style do I like?",
]

print("=== Semantic Memory Recall ===")
for query in SESSION_2_QUERIES:
    # limit controls recall vs precision: higher k returns more memories but may dilute relevance
    results = mem.search(query, user_id=USER_ID, limit=3)
    memories = results.get("results", results) if isinstance(results, dict) else results
    print(f"\nQ: {query}")
    for m in memories:
        text = m.get('memory', m.get('text', str(m)))
        # score is cosine similarity — closer to 1.0 means the stored fact closely matches the query embedding
        score = m.get('score', m.get('similarity', ''))
        score_str = f" [score={score:.3f}]" if isinstance(score, float) else ""
        print(f"  → {text}{score_str}")

---
## Part 4 — LangGraph Pipeline: recall → respond → store

```
START → recall → respond → store → END
```

In [ ]:
from typing import TypedDict
from langchain_core.messages import HumanMessage
from langgraph.graph import END, START, StateGraph

# TypedDict gives LangGraph typed access to state keys — no runtime overhead, just static typing
class MemState(TypedDict):
    user_id: str; query: str; recalled: list
    answer: str; messages: list

def recall_node(state):
    results = mem.search(state["query"], user_id=state["user_id"], limit=5)
    memories = results.get("results", results) if isinstance(results, dict) else results
    return {"recalled": memories}

def respond_node(state):
    context = "\n".join(
        f"- {m.get('memory', m.get('text', ''))}" for m in state["recalled"]
    )
    prompt = (
        f"Memories about the user:\n{context or '(none)'}\n\n"
        f"User: {state['query']}\n"
        "Answer based on the memories if relevant, otherwise say you don't know."
    )
    answer = llm.invoke([HumanMessage(content=prompt)]).content
    return {"answer": answer}

def store_node(state):
    if state.get("messages"):
        mem.add(state["messages"], user_id=state["user_id"])
    return {}

# StateGraph takes the TypedDict class as schema — every node receives and returns a dict slice
g = StateGraph(MemState)
for name, fn in [("recall", recall_node), ("respond", respond_node), ("store", store_node)]:
    g.add_node(name, fn)
g.add_edge(START, "recall")
g.add_edge("recall", "respond")
g.add_edge("respond", "store")
g.add_edge("store", END)
# compile() locks the graph topology and makes it invokable — also where you'd attach a checkpointer
# Alternative: app = g.compile(checkpointer=MemorySaver()) for in-process short-term memory
app = g.compile()

---
## Part 5 — Two-Session Demo

In [ ]:
print("=== Session 2: Recall from stored memories ===")
for query in SESSION_2_QUERIES:
    init = {
        "user_id": USER_ID, "query": query,
        "recalled": [], "answer": "", "messages": []
    }
    # invoke() runs the full recall → respond → store pipeline synchronously
    # Alternative: use app.stream(init) to yield state after each node for step-by-step debugging
    result = app.invoke(init)
    print(f"Q: {query}")
    print(f"Recalled {len(result['recalled'])} memories")
    print(f"A: {result['answer'].strip()}")
    print()

print("=== Add new memories in Session 2 ===")
new_messages = [
    {"role": "user",      "content": "I'm currently learning Rust."},
    {"role": "assistant", "content": "Great! Rust is excellent for systems programming."},
]
store_init = {"user_id": USER_ID, "query": "", "recalled": [], "answer": "", "messages": new_messages}
app.invoke(store_init)

# Verify new memory was stored
rust_results = mem.search("What languages is Alice learning?", user_id=USER_ID, limit=3)
rust_memories = rust_results.get("results", rust_results) if isinstance(rust_results, dict) else rust_results
print("Rust memory check:")
for m in rust_memories:
    print(f"  → {m.get('memory', m.get('text', str(m)))}")

---
### Exercise 1 — Multi-User Isolation
Create a second user `"demo-user-002"` with different preferences (e.g., Java, verbose code). Run the same queries for both users. Confirm memories are isolated per `user_id`.

### Exercise 2 — Memory Deletion
Call `mem.delete_all(user_id=USER_ID)` and re-run the recall queries. What happens to the answers when memories are cleared?

In [ ]:
# Exercise 1 — multi-user isolation
USER_2 = "demo-user-002"
user2_messages = [
    {"role": "user",      "content": "Hi, I'm Bob. I prefer Java over Python."},
    {"role": "assistant", "content": "Got it, Bob! Java it is."},
]
mem.add(user2_messages, user_id=USER_2)

for uid, name in [(USER_ID, "Alice"), (USER_2, "Bob")]:
    r = mem.search("What programming language does this user prefer?", user_id=uid, limit=2)
    mems = r.get("results", r) if isinstance(r, dict) else r
    top = mems[0].get('memory', '') if mems else '(none)'
    print(f"{name} ({uid}): {top}")

# Exercise 2 — delete and re-check
# Uncomment to run:
# mem.delete_all(user_id=USER_ID)
# r = mem.search("What language does Alice prefer?", user_id=USER_ID, limit=3)
# print(f"After delete: {r}")

---
*Workshop complete. Next: example 70 — Prompt Injection Defense.*